# Crime Analysis Pipeline

In [1]:
# Import modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## Ingestion

### Lookup Table


Link for LAD <-> PFA data: __https://ckan.publishing.service.gov.uk/dataset/local-authority-district-to-community-safety-partnership-to-pfa-april-2025-lookup-in-ew/resource/e8cc60d8-f3bb-4a29-ad58-821247d88d95__

Link for LSOA <-> LAD data: __https://ckan.publishing.service.gov.uk/dataset/lsoa-2021-to-electoral-ward-2024-to-lad-2024-best-fit-lookup-in-ew__

In [2]:
# Import LAD <-> PAF raw data
raw_lad_pfr = pd.read_csv('../Data/Raw/lookup/lad-pfa.csv')

raw_lad_pfr.head(5)

,LAD25CD,LAD25NM,CSP25CD,CSP25NM,PFA25CD,PFA25NM,ObjectId
0,E06000058,"Bournemouth, Christchurch and Poole",E22000367,Dorset,E23000039,Dorset,1
1,E06000059,Dorset,E22000367,Dorset,E23000039,Dorset,2
2,E06000060,Buckinghamshire,E22000303,Aylesbury Vale,E23000029,Thames Valley,3
3,E06000060,Buckinghamshire,E22000306,Chiltern,E23000029,Thames Valley,4
4,E06000060,Buckinghamshire,E22000311,South Bucks,E23000029,Thames Valley,5


In [3]:
# Import LSOA <-> LAD data:
raw_lsoa_lad = pd.read_csv('../Data/Raw/lookup/lsoa-lad.csv')

raw_lsoa_lad.head(5)

,LSOA21CD,LSOA21NM,LSOA21NMW,WD24CD,WD24NM,WD24NMW,LAD24CD,LAD24NM,LAD24NMW,ObjectId
0,E01012000,Hartlepool 007E,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,1
1,E01011964,Hartlepool 007B,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,2
2,E01011999,Hartlepool 007D,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,3
3,E01011967,Hartlepool 007C,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,4
4,E01011951,Hartlepool 007A,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,5


### Population Data

Link for population data: __https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/lowersuperoutputareamidyearpopulationestimates__

<div class="alert alert-block alert-warning">
<b>Warning:</b> This section may take some time to import, there are 35,000 rows per year. It usually takes ~1min 30seconds to complete each import.
</div>

In [4]:
# Import Population Data Raw
raw_pop_2022 = pd.read_excel(f'../Data/Raw/population/population.xlsx', sheet_name='Mid-2022 LSOA 2021', skiprows=3, usecols=['LAD 2023 Code', 'LAD 2023 Name', 'LSOA 2021 Code', 'LSOA 2021 Name', 'Total'])

raw_pop_2022.head()

,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1876
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1117
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1260
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1635
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,1984


In [5]:
raw_pop_2023 = pd.read_excel(f'../Data/Raw/population/population.xlsx', sheet_name='Mid-2023 LSOA 2021', skiprows=3, usecols=['LAD 2023 Code', 'LAD 2023 Name', 'LSOA 2021 Code', 'LSOA 2021 Name', 'Total'])

raw_pop_2023.head()

,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1925
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1177
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1320
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1670
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2075


In [6]:
raw_pop_2024 = pd.read_excel(f'../Data/Raw/population/population.xlsx', sheet_name='Mid-2024 LSOA 2021', skiprows=3, usecols=['LAD 2023 Code', 'LAD 2023 Name', 'LSOA 2021 Code', 'LSOA 2021 Name', 'Total'])

raw_pop_2024.head()

,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1898
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1247
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1393
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1669
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2303


### Deprivation Data

Link for Deprivation data: __https://www.gov.uk/csv-preview/691ded56d140bbbaa59a2a7d/File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv__

In [122]:
# Import Deprivation Data Raw
raw_depr = pd.read_csv(f'../Data/Raw/deprivation/deprivation.csv')

raw_depr.head()


,LSOA code (2021),LSOA name (2021),Local Authority District code (2024),Local Authority District name (2024),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Score,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2022,Dependent Children aged 0-15: mid 2022,Older population aged 60 and over: mid 2022,Working age population 18-66 (for use with Employment Deprivation Domain): mid 2022
0,E01000001,City of London 001A,E09000001,City of London,8.742,26525,8,0.013,33730,10,...,1.207,1105,1,1.414,1586,1,1795,149,520,1248
1,E01000002,City of London 001B,E09000001,City of London,4.722,31203,10,0.018,33669,10,...,0.355,9591,3,1.839,592,1,1671,81,387,1324
2,E01000003,City of London 001C,E09000001,City of London,9.250,25913,8,0.107,25167,8,...,0.318,10175,4,1.679,903,1,1896,136,432,1469
3,E01000005,City of London 001E,E09000001,City of London,19.884,14807,5,0.211,14836,5,...,0.012,15502,5,2.065,303,1,1737,177,160,1448
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,25.307,10917,4,0.343,7519,3,...,0.399,8934,3,0.400,9136,3,1837,397,225,1260


### Crime Severity Data

In [8]:
# Import crime severity categorised data set
sev = pd.read_csv(f'../Data/Processed/crime-severity-raw/crime-severity-categorised.csv')

sev.head()

,Crime Index,Offence,Weight,Crime Category
0,"1, 4.1/10/2",Homicide,"7,979",Violence and sexual offences
1,2,Attempted murder,"4,663",Violence and sexual offences
2,4.3,Intentional destruction of viable unborn child,15,Violence and sexual offences
3,4.4,Causing death or serious injury by dangerous d...,"1,092",Violence and sexual offences
4,4.6,Causing death by careless driving when under t...,"1,595",Violence and sexual offences


### Crime Data

In [9]:
## Import one months worth of data
police_region = 'merseyside'
year_month = '2026-03' # Get the most recent data available

mssd = pd.read_csv(f'../Data/Raw/crime-data/{police_region}/{year_month}-{police_region}-street.csv')

mssd.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2026-03,Merseyside Police,Merseyside Police,-2.871827,53.489763,On or near Gilescroft Avenue,E01006448,Knowsley 001A,Anti-social behaviour,NaN,NaN
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,2026-03,Merseyside Police,Merseyside Police,-2.874541,53.485420,On or near Harleston Road,E01006448,Knowsley 001A,Criminal damage and arson,Unable to prosecute suspect,NaN
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,2026-03,Merseyside Police,Merseyside Police,-2.872892,53.488785,On or near Brook Hey Drive,E01006448,Knowsley 001A,Criminal damage and arson,Investigation complete; no suspect identified,NaN
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,2026-03,Merseyside Police,Merseyside Police,-2.870190,53.485658,On or near Darmond Road,E01006448,Knowsley 001A,Drugs,Under investigation,NaN
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,2026-03,Merseyside Police,Merseyside Police,-2.874261,53.490168,On or near Kenbury Close,E01006448,Knowsley 001A,Other theft,Investigation complete; no suspect identified,NaN


## Cleaning & Validation

### Lookup Table

**LAD <-> PFA conversion database**

In [10]:
# Check data types and overall size
raw_lad_pfr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 332 entries, 0 to 331
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   LAD25CD   332 non-null    object
 1   LAD25NM   332 non-null    object
 2   CSP25CD   332 non-null    object
 3   CSP25NM   332 non-null    object
 4   PFA25CD   332 non-null    object
 5   PFA25NM   332 non-null    object
 6   ObjectId  332 non-null    int64 
dtypes: int64(1), object(6)
memory usage: 18.3+ KB


***
All datatypes are correct.
***

In [11]:
# Check for null values
raw_lad_pfr.isnull().sum()

LAD25CD     0
LAD25NM     0
CSP25CD     0
CSP25NM     0
PFA25CD     0
PFA25NM     0
ObjectId    0
dtype: int64

***
No data is null.
***

In [12]:
# Check for duplicated data
raw_lad_pfr.duplicated().sum()

np.int64(0)

***
No duplicated rows.
***

***
**Remove Unneeded Columns**  
Needed Columns:
- LAD Code
- LAD Name
- PFA Code
- PFA Name
***

In [13]:
lad_pfr = raw_lad_pfr[['LAD25CD', 'LAD25NM', 'PFA25CD', 'PFA25NM']]

# Rename columns to easier names

lad_pfr = lad_pfr.rename(columns={
    'LAD25CD': 'lad_code',
    'LAD25NM': 'lad_name',
    'PFA25CD': 'pfa_code',
    'PFA25NM': 'pfa_name'
})

lad_pfr.head()

,lad_code,lad_name,pfa_code,pfa_name
0,E06000058,"Bournemouth, Christchurch and Poole",E23000039,Dorset
1,E06000059,Dorset,E23000039,Dorset
2,E06000060,Buckinghamshire,E23000029,Thames Valley
3,E06000060,Buckinghamshire,E23000029,Thames Valley
4,E06000060,Buckinghamshire,E23000029,Thames Valley


**LSOA <-> LAD conversion database**

In [14]:
# Check data types and overall size
raw_lsoa_lad.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   LSOA21CD   35672 non-null  object
 1   LSOA21NM   35672 non-null  object
 2   LSOA21NMW  1917 non-null   object
 3   WD24CD     35672 non-null  object
 4   WD24NM     35672 non-null  object
 5   WD24NMW    1917 non-null   object
 6   LAD24CD    35672 non-null  object
 7   LAD24NM    35672 non-null  object
 8   LAD24NMW   1917 non-null   object
 9   ObjectId   35672 non-null  int64 
dtypes: int64(1), object(9)
memory usage: 2.7+ MB


***
All datatypes are correct.
***

In [15]:
# Check for null values
raw_lsoa_lad.isnull().sum()

LSOA21CD         0
LSOA21NM         0
LSOA21NMW    33755
WD24CD           0
WD24NM           0
WD24NMW      33755
LAD24CD          0
LAD24NM          0
LAD24NMW     33755
ObjectId         0
dtype: int64

***
Only useless data is null, can be ignored as we will be dropping to essentials shortly.
***

In [16]:
# Check for duplicated data
raw_lsoa_lad.duplicated().sum()

np.int64(0)

***
No duplicated rows.
***

***
**Remove Unneeded Columns**  
Needed Columns:
- LSOA Code
- LSOA Name
- LAD Code
- LAD Name
***

In [17]:
lsoa_lad = raw_lsoa_lad[['LSOA21CD', 'LSOA21NM', 'LAD24CD', 'LAD24NM']]

# Rename columns to easier names

lsoa_lad = lsoa_lad.rename(columns={
    'LSOA21CD': 'lsoa_code',
    'LSOA21NM': 'lsoa_name',
    'LAD24CD': 'lad_code',
    'LAD24NM': 'lad_name'
})

lsoa_lad.head()

,lsoa_code,lsoa_name,lad_code,lad_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool
1,E01011964,Hartlepool 007B,E06000001,Hartlepool
2,E01011999,Hartlepool 007D,E06000001,Hartlepool
3,E01011967,Hartlepool 007C,E06000001,Hartlepool
4,E01011951,Hartlepool 007A,E06000001,Hartlepool


### Population

In [18]:
raw_pop_2022.info

<bound method DataFrame.info of       LAD 2023 Code   LAD 2023 Name LSOA 2021 Code       LSOA 2021 Name  Total
0         E06000001      Hartlepool      E01011949      Hartlepool 009A   1876
1         E06000001      Hartlepool      E01011950      Hartlepool 008A   1117
2         E06000001      Hartlepool      E01011951      Hartlepool 007A   1260
3         E06000001      Hartlepool      E01011952      Hartlepool 002A   1635
4         E06000001      Hartlepool      E01011953      Hartlepool 002B   1984
...             ...             ...            ...                  ...    ...
35667     W06000024  Merthyr Tydfil      W01001324  Merthyr Tydfil 003E   1888
35668     W06000024  Merthyr Tydfil      W01001898  Merthyr Tydfil 008F   1452
35669     W06000024  Merthyr Tydfil      W01001959  Merthyr Tydfil 005E   1547
35670     W06000024  Merthyr Tydfil      W01001960  Merthyr Tydfil 005F   1481
35671     W06000024  Merthyr Tydfil      W01001961  Merthyr Tydfil 006G   2321

[35672 rows x 5 col

In [19]:
raw_pop_2023.info

<bound method DataFrame.info of       LAD 2023 Code   LAD 2023 Name LSOA 2021 Code       LSOA 2021 Name  Total
0         E06000001      Hartlepool      E01011949      Hartlepool 009A   1925
1         E06000001      Hartlepool      E01011950      Hartlepool 008A   1177
2         E06000001      Hartlepool      E01011951      Hartlepool 007A   1320
3         E06000001      Hartlepool      E01011952      Hartlepool 002A   1670
4         E06000001      Hartlepool      E01011953      Hartlepool 002B   2075
...             ...             ...            ...                  ...    ...
35667     W06000024  Merthyr Tydfil      W01001324  Merthyr Tydfil 003E   1844
35668     W06000024  Merthyr Tydfil      W01001898  Merthyr Tydfil 008F   1437
35669     W06000024  Merthyr Tydfil      W01001959  Merthyr Tydfil 005E   1555
35670     W06000024  Merthyr Tydfil      W01001960  Merthyr Tydfil 005F   1467
35671     W06000024  Merthyr Tydfil      W01001961  Merthyr Tydfil 006G   2333

[35672 rows x 5 col

In [20]:
raw_pop_2024.info

<bound method DataFrame.info of       LAD 2023 Code   LAD 2023 Name LSOA 2021 Code       LSOA 2021 Name  Total
0         E06000001      Hartlepool      E01011949      Hartlepool 009A   1898
1         E06000001      Hartlepool      E01011950      Hartlepool 008A   1247
2         E06000001      Hartlepool      E01011951      Hartlepool 007A   1393
3         E06000001      Hartlepool      E01011952      Hartlepool 002A   1669
4         E06000001      Hartlepool      E01011953      Hartlepool 002B   2303
...             ...             ...            ...                  ...    ...
35667     W06000024  Merthyr Tydfil      W01001324  Merthyr Tydfil 003E   1848
35668     W06000024  Merthyr Tydfil      W01001898  Merthyr Tydfil 008F   1441
35669     W06000024  Merthyr Tydfil      W01001959  Merthyr Tydfil 005E   1528
35670     W06000024  Merthyr Tydfil      W01001960  Merthyr Tydfil 005F   1453
35671     W06000024  Merthyr Tydfil      W01001961  Merthyr Tydfil 006G   2356

[35672 rows x 5 col

In [21]:
raw_pop_2022.isnull().sum()

LAD 2023 Code     0
LAD 2023 Name     0
LSOA 2021 Code    0
LSOA 2021 Name    0
Total             0
dtype: int64

In [22]:
raw_pop_2023.isnull().sum()

LAD 2023 Code     0
LAD 2023 Name     0
LSOA 2021 Code    0
LSOA 2021 Name    0
Total             0
dtype: int64

In [23]:
raw_pop_2024.isnull().sum()

LAD 2023 Code     0
LAD 2023 Name     0
LSOA 2021 Code    0
LSOA 2021 Name    0
Total             0
dtype: int64

In [24]:
raw_pop_2022.duplicated().sum()

np.int64(0)

In [25]:
raw_pop_2023.duplicated().sum()

np.int64(0)

In [26]:
raw_pop_2024.duplicated().sum()

np.int64(0)

***
All data is incredibly clean: correct datatype, no null values, and no duplicates
***

### Deprivation

In [123]:
# Check data types and overall size
raw_depr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33755 entries, 0 to 33754
Data columns (total 56 columns):
 #   Column                                                                                              Non-Null Count  Dtype  
---  ------                                                                                              --------------  -----  
 0   LSOA code (2021)                                                                                    33755 non-null  object 
 1   LSOA name (2021)                                                                                    33755 non-null  object 
 2   Local Authority District code (2024)                                                                33755 non-null  object 
 3   Local Authority District name (2024)                                                                33755 non-null  object 
 4   Index of Multiple Deprivation (IMD) Score                                                           33755 non-nu

***
**Only need columns that are related to data:**  
Keep:  
LSOA Code
Index of Multiple Deprivation (IMD) Score  
Income Score (rate)  
Employment Score (rate)  
Education, Skills and Training Score   
Barriers to Housing and Services Score  
  

Drop all others.
***

In [124]:
depr = raw_depr[[
    'LSOA code (2021)', 
    'Index of Multiple Deprivation (IMD) Score',
    'Income Score (rate)',
    'Employment Score (rate)',
    'Education, Skills and Training Score',
    'Barriers to Housing and Services Score'
]]

depr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33755 entries, 0 to 33754
Data columns (total 6 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   LSOA code (2021)                           33755 non-null  object 
 1   Index of Multiple Deprivation (IMD) Score  33755 non-null  float64
 2   Income Score (rate)                        33755 non-null  float64
 3   Employment Score (rate)                    33755 non-null  float64
 4   Education, Skills and Training Score       33755 non-null  float64
 5   Barriers to Housing and Services Score     33755 non-null  float64
dtypes: float64(5), object(1)
memory usage: 1.5+ MB


***
Data types are all correct.  
We only need to keep rows relating to the districts that we have crime data on.  
We have the LAD code of each  region, we need a new column that tells us its PFA to tell us which police force has jurisdiction over that LAD.
  
**Conclusion:** Create a PFA Code column, by merging the lad-pfa database. To be done in next section.
***

In [125]:
# Check for null values
depr.isnull().sum()

LSOA code (2021)                             0
Index of Multiple Deprivation (IMD) Score    0
Income Score (rate)                          0
Employment Score (rate)                      0
Education, Skills and Training Score         0
Barriers to Housing and Services Score       0
dtype: int64

***
No nulls in data.
***

In [126]:
# Check for duplicated data
depr.duplicated().sum()

np.int64(0)

***
No duplicated data
***

### Crime Severity

In [32]:
# Check data types and overall size
sev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Crime Index     245 non-null    object
 1   Offence         250 non-null    object
 2   Weight          250 non-null    object
 3   Crime Category  242 non-null    object
dtypes: object(4)
memory usage: 7.9+ KB


***
**Data types should be:**  
Crime Index :   object  
Offense:        object  
Weight:         int64  
Crime Category: object  
  
**Conclusion:** Change weight to an integer
***

In [33]:
# Check for nulls
sev.isnull().sum()

Crime Index       5
Offence           0
Weight            0
Crime Category    8
dtype: int64

***
**Crime Index has null values.**
- Looking at the data, it serves no purpose for the reason we need the datasheet for. It is worth dropping the column.  
**Conclusion**: Drop the column

***

**Crime Category has null values.**
- Crime category cannot be dropped, it is required.  
- Set crime category nulls to 'No Category', such that they are known as null.   
- Looking at the data, these offenses are related to cyber risks and hacking. These will not be found within the crime dataset, as it only logs in person activities.  
**Conclusion**: Set null values to 'No Category'
***

In [34]:
# Check for duplicates
print(sev.duplicated().sum())

0


***
**There are 3 duplicated rows in the dataset**  
No reason to keep the duplicated rows in the dataset, it is repeated data, and will skew averages.  
**Conclusion:** Remove dupllicated rows
***
***

In [35]:
# Change weight to an integer
sev['Weight'] = sev['Weight'].astype(str).str.replace(',', '', regex=False)
sev['Weight'] = pd.to_numeric(sev['Weight'], errors='coerce')

print(f'Weight datatype: {sev['Weight'].dtype}')
print(f'Number of nulls in Weight: {sev['Weight'].isnull().sum()}')
print(f'Sample of Weight:')
display(sev['Weight'].sample(3))

## 

Weight datatype: int64
Number of nulls in Weight: 0
Sample of Weight:


210     86
37     107
96     117
Name: Weight, dtype: int64

In [36]:
#Drop Crime Index
sev = sev.drop(columns=['Crime Index'])

print(f'Sample of severance weighting:')
display(sev.sample(3))

Sample of severance weighting:


,Offence,Weight,Crime Category
136,Possession of other weapons,58,Possession of weapons
245,Other regulatory fraud,95,Other crime
137,Possession of article with blade or point,55,Possession of weapons


In [37]:
# Set null values of Crime Category to 'No Category'
sev['Crime Category'] = sev['Crime Category'].fillna('No Category')

print(sev['Crime Category'].value_counts())

Crime Category
Other crime                     93
Violence and sexual offences    83
Burglary                        16
Criminal damage and arson       12
Public order                     9
Other theft                      8
No Category                      8
Possession of weapons            7
Drugs                            5
Vehicle crime                    4
Robbery                          2
Theft from the person            1
Bicycle Theft                    1
Shoplifting                      1
Name: count, dtype: int64


In [38]:
# Remove duplicated rows
sev = sev.drop_duplicates()
print(sev.duplicated().sum())

0


***
***
Data in crime severity is now clean.

In [39]:
sev.info()

<class 'pandas.core.frame.DataFrame'>
Index: 247 entries, 0 to 249
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Offence         247 non-null    object
 1   Weight          247 non-null    int64 
 2   Crime Category  247 non-null    object
dtypes: int64(1), object(2)
memory usage: 7.7+ KB


In [40]:
sev.isnull().sum()

Offence           0
Weight            0
Crime Category    0
dtype: int64

### Crime Data

In [41]:
mssd.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2026-03,Merseyside Police,Merseyside Police,-2.871827,53.489763,On or near Gilescroft Avenue,E01006448,Knowsley 001A,Anti-social behaviour,NaN,NaN
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,2026-03,Merseyside Police,Merseyside Police,-2.874541,53.485420,On or near Harleston Road,E01006448,Knowsley 001A,Criminal damage and arson,Unable to prosecute suspect,NaN
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,2026-03,Merseyside Police,Merseyside Police,-2.872892,53.488785,On or near Brook Hey Drive,E01006448,Knowsley 001A,Criminal damage and arson,Investigation complete; no suspect identified,NaN
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,2026-03,Merseyside Police,Merseyside Police,-2.870190,53.485658,On or near Darmond Road,E01006448,Knowsley 001A,Drugs,Under investigation,NaN
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,2026-03,Merseyside Police,Merseyside Police,-2.874261,53.490168,On or near Kenbury Close,E01006448,Knowsley 001A,Other theft,Investigation complete; no suspect identified,NaN


In [42]:
# Looking at data
mssd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13356 entries, 0 to 13355
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               12058 non-null  object 
 1   Month                  13356 non-null  object 
 2   Reported by            13356 non-null  object 
 3   Falls within           13356 non-null  object 
 4   Longitude              13356 non-null  float64
 5   Latitude               13356 non-null  float64
 6   Location               13356 non-null  object 
 7   LSOA code              13356 non-null  object 
 8   LSOA name              13356 non-null  object 
 9   Crime type             13356 non-null  object 
 10  Last outcome category  12058 non-null  object 
 11  Context                0 non-null      float64
dtypes: float64(3), object(9)
memory usage: 1.2+ MB


***
Data types are **not** all correct.  
Month should be a date.
Context should be an object. (also it will be dropped so doesn't matter.)

**Conclusion:** Set month to be held as a date.
***

In [43]:
mssd.isnull().sum()

Crime ID                  1298
Month                        0
Reported by                  0
Falls within                 0
Longitude                    0
Latitude                     0
Location                     0
LSOA code                    0
LSOA name                    0
Crime type                   0
Last outcome category     1298
Context                  13356
dtype: int64

***
**Crime ID has null values** - This means we need to create a new primary key for this database.  
As each database is categorised by its location and its year and month, we will use that in its primary key.  
eg: mers_2026_03_00001 - This allows for up to 100,000 crimes per month per police region.  
  
Last outcome category has null values. This will eventually be dropped, so it is not a worry.  
context is fully null, it will eventually be dropped, so it is not a worry.
  
**Conclusion:** Create a new Crime ID column, drop last outcome and context.
***

In [44]:
mssd.duplicated().sum()

np.int64(399)

***
Duplicate values are present, this includes crime id and all columns - so data has just been entered twice onto the database.  
These can be dropped easily.  
  
**Conclusion:** Drop duplicate values.
***

Columns that provide useful information for final aggregated database:
- Crime ID (new)
- Month (Year/Month)
- Latitude
- Longnitude
- LSOA Code
- LSOA Name
- Crime Category

Drop all others

In [45]:
# Drop useless columns

mssd = mssd[['Crime ID', 'LSOA code', 'LSOA name', 'Month', 'Latitude', 'Longitude', 'Crime type']]

mssd.head()

,Crime ID,LSOA code,LSOA name,Month,Latitude,Longitude,Crime type
0,NaN,E01006448,Knowsley 001A,2026-03,53.489763,-2.871827,Anti-social behaviour
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,E01006448,Knowsley 001A,2026-03,53.485420,-2.874541,Criminal damage and arson
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,E01006448,Knowsley 001A,2026-03,53.488785,-2.872892,Criminal damage and arson
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,E01006448,Knowsley 001A,2026-03,53.485658,-2.870190,Drugs
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,E01006448,Knowsley 001A,2026-03,53.490168,-2.874261,Other theft


In [46]:
# Rename columns
mssd = mssd.rename(columns={
    'Crime ID': 'old_crime_id',
    'Month': 'date',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'LSOA code': 'lsoa_code',
    'LSOA name': 'lsoa_name',
    'Crime type': 'crime_cat'
})

mssd.head()

,old_crime_id,lsoa_code,lsoa_name,date,latitude,longitude,crime_cat
0,NaN,E01006448,Knowsley 001A,2026-03,53.489763,-2.871827,Anti-social behaviour
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,E01006448,Knowsley 001A,2026-03,53.485420,-2.874541,Criminal damage and arson
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,E01006448,Knowsley 001A,2026-03,53.488785,-2.872892,Criminal damage and arson
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,E01006448,Knowsley 001A,2026-03,53.485658,-2.870190,Drugs
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,E01006448,Knowsley 001A,2026-03,53.490168,-2.874261,Other theft


In [47]:
# Drop duplicate rows
print(f'row count before dropping: {mssd.shape[0]}')

mssd = mssd.drop_duplicates()
print(f'number of duplicates after dropping: {mssd.duplicated().sum()}')

print(f'row count after dropping: {mssd.shape[0]}')

row count before dropping: 13356
number of duplicates after dropping: 0
row count after dropping: 12957


In [48]:
# Change month to date
mssd['date'] = pd.to_datetime(mssd['date'], format='%Y-%m')

mssd.head()

,old_crime_id,lsoa_code,lsoa_name,date,latitude,longitude,crime_cat
0,NaN,E01006448,Knowsley 001A,2026-03-01,53.489763,-2.871827,Anti-social behaviour
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,E01006448,Knowsley 001A,2026-03-01,53.485420,-2.874541,Criminal damage and arson
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,E01006448,Knowsley 001A,2026-03-01,53.488785,-2.872892,Criminal damage and arson
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,E01006448,Knowsley 001A,2026-03-01,53.485658,-2.870190,Drugs
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,E01006448,Knowsley 001A,2026-03-01,53.490168,-2.874261,Other theft


In [49]:
mssd.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12957 entries, 0 to 13355
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   old_crime_id  12058 non-null  object        
 1   lsoa_code     12957 non-null  object        
 2   lsoa_name     12957 non-null  object        
 3   date          12957 non-null  datetime64[ns]
 4   latitude      12957 non-null  float64       
 5   longitude     12957 non-null  float64       
 6   crime_cat     12957 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(4)
memory usage: 809.8+ KB


***
Dataset is now clean to start engineering and transformation.
***

## Feature Engineering & Transformation

#### Lookup Table

***
This section will merge the lsoa-lad and lad-pfa datasets to create an aggregated lookup table, where each location can be found.
***

In [50]:
# merge on lad code
# final table columns: LSOA Code | LSOA Name | LAD Code | LAD Name | PFA Code | PFA Name

lookup_table = pd.merge(lsoa_lad, lad_pfr, how='outer', on=['lad_code'], indicator=True)

lookup_table.head(5)

,lsoa_code,lsoa_name,lad_code,lad_name_x,lad_name_y,pfa_code,pfa_name,_merge
0,E01012000,Hartlepool 007E,E06000001,Hartlepool,Hartlepool,E23000013,Cleveland,both
1,E01011964,Hartlepool 007B,E06000001,Hartlepool,Hartlepool,E23000013,Cleveland,both
2,E01011999,Hartlepool 007D,E06000001,Hartlepool,Hartlepool,E23000013,Cleveland,both
3,E01011967,Hartlepool 007C,E06000001,Hartlepool,Hartlepool,E23000013,Cleveland,both
4,E01011951,Hartlepool 007A,E06000001,Hartlepool,Hartlepool,E23000013,Cleveland,both


In [51]:
lookup_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38792 entries, 0 to 38791
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   lsoa_code   38790 non-null  object  
 1   lsoa_name   38790 non-null  object  
 2   lad_code    38792 non-null  object  
 3   lad_name_x  38790 non-null  object  
 4   lad_name_y  38301 non-null  object  
 5   pfa_code    38301 non-null  object  
 6   pfa_name    38301 non-null  object  
 7   _merge      38792 non-null  category
dtypes: category(1), object(7)
memory usage: 2.1+ MB


In [52]:
lookup_table.isnull().sum()

lsoa_code       2
lsoa_name       2
lad_code        0
lad_name_x      2
lad_name_y    491
pfa_code      491
pfa_name      491
_merge          0
dtype: int64

In [53]:
lookup_table[lookup_table['lsoa_code'].isna()]

,lsoa_code,lsoa_name,lad_code,lad_name_x,lad_name_y,pfa_code,pfa_name,_merge
31879,NaN,NaN,E08000038,NaN,Barnsley,E23000011,South Yorkshire,right_only
31880,NaN,NaN,E08000039,NaN,Sheffield,E23000011,South Yorkshire,right_only


In [54]:
lookup_table[lookup_table['pfa_code'].isna()]['lad_code'].value_counts()

lad_code
E08000019    343
E08000016    148
Name: count, dtype: int64

***
Two missing values present in the lsoa <-> lad database - for Barnsley and Sheffield in South Yorkshire.    
Furthermore, there are two missing entries in the lad <-> pfa database - again for Barnsley and Sheffield in South Yorkshire. (E...016-E...019)
  
This poses an issue, as these areas are within the crime data this project is looking at.  
  
After doing some reseaching online, the two sectors we are getting issues in were both updated in 2025, according to these links:  
Barnsley: __https://www.ons.gov.uk/explore-local-statistics/areas/E08000038-barnsley__
Sheffield: __https://www.ons.gov.uk/explore-local-statistics/areas/E08000039-sheffield__

This means that these are the same issues, and can be merged to be the same data.

I need to ensure that whenever I am using LAD data before 2025, I update the LAD to E08000039 from the outdated E08000019 and similarly I update to E08000038 from the outdated E08000016.

**Conclusion:** In the lsoa->lad dataset, set any instances of E08000019 or E08000016 to their updated counterparts. Then the datasets can be remerged to create a clean lookup table
***

In [55]:
# Change lad->pfr dataset outdated data

# Update old Sheffield LAD code to new Sheffield LAD code
lsoa_lad.loc[lsoa_lad['lad_code'] == 'E08000016', 'lad_code'] = 'E08000038'

# Update old Barnsley LAD code to new Barnsley LAD code
lsoa_lad.loc[lsoa_lad['lad_code'] == 'E08000019', 'lad_code'] = 'E08000039'

lsoa_lad[lsoa_lad['lad_code'].isin(['E08000038', 'E08000039'])]

# Updated succesfully!

,lsoa_code,lsoa_name,lad_code,lad_name
24051,E01007429,Barnsley 024C,E08000038,Barnsley
24054,E01007444,Barnsley 012F,E08000038,Barnsley
24058,E01007428,Barnsley 024B,E08000038,Barnsley
24059,E01007334,Barnsley 009A,E08000038,Barnsley
24061,E01007382,Barnsley 019A,E08000038,Barnsley
...,...,...,...,...
25184,E01008134,Sheffield 005B,E08000039,Sheffield
25187,E01007888,Sheffield 003A,E08000039,Sheffield
25190,E01007899,Sheffield 003E,E08000039,Sheffield
25193,E01007901,Sheffield 003G,E08000039,Sheffield


***
**Now actually merging the datasets together!**

We made it!

In [56]:
# merge on lad code
# final table columns: LSOA Code | LSOA Name | LAD Code | LAD Name | PFA Code | PFA Name

lookup_table = pd.merge(lsoa_lad, lad_pfr, how='left', on=['lad_code', 'lad_name'])

lookup_table.head(5)

,lsoa_code,lsoa_name,lad_code,lad_name,pfa_code,pfa_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool,E23000013,Cleveland
1,E01011964,Hartlepool 007B,E06000001,Hartlepool,E23000013,Cleveland
2,E01011999,Hartlepool 007D,E06000001,Hartlepool,E23000013,Cleveland
3,E01011967,Hartlepool 007C,E06000001,Hartlepool,E23000013,Cleveland
4,E01011951,Hartlepool 007A,E06000001,Hartlepool,E23000013,Cleveland


In [57]:
lookup_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38790 entries, 0 to 38789
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   lsoa_code  38790 non-null  object
 1   lsoa_name  38790 non-null  object
 2   lad_code   38790 non-null  object
 3   lad_name   38790 non-null  object
 4   pfa_code   38790 non-null  object
 5   pfa_name   38790 non-null  object
dtypes: object(6)
memory usage: 1.8+ MB


***
All values are the correct datatypes
***

In [58]:
lookup_table.isnull().sum()

lsoa_code    0
lsoa_name    0
lad_code     0
lad_name     0
pfa_code     0
pfa_name     0
dtype: int64

***
No null data
***

In [59]:
lookup_table.duplicated().sum()

np.int64(3118)

***
3118 duplicated rows, drop.
***

In [60]:
# Drop duplicate rows
print(f'row count before dropping: {lookup_table.shape[0]}')

lookup_table = lookup_table.drop_duplicates()
print(f'number of duplicates after dropping: {lookup_table.duplicated().sum()}')

print(f'row count after dropping: {lookup_table.shape[0]}')

row count before dropping: 38790
number of duplicates after dropping: 0
row count after dropping: 35672


***
Now we have a clean lookup database, we can export it to be used later.
***

In [61]:
## Export the code to processed folder as a csv
lookup_table.to_csv('../Data/Processed/lookup-table.csv', index=False)
# Keep the lad_to pfr for databases holding only lad, as to avoid confusion
lad_pfr.to_csv('../Data/Processed/lad-pfr.csv', index=False)

print(f'lookup-table.csv File successfully created: {Path('../Data/Processed/lookup-table.csv').exists()}')
print(f'lad-pfr.csv File successfully created: {Path('../Data/Processed/lad-pfr.csv').exists()}')

lookup-table.csv File successfully created: True
lad-pfr.csv File successfully created: True


#### Population

***
This section will create estimated totals for the populations of LSOA's in 2025 and 2026, then produce a processed database from the now 5 population databases. It will have the columns:  
LSOA Code | LSOA Name | LAD Code | LAD Name | PFR Code | PFR Name |  Total Population_2022/3/4/5/6 (as separate columns)
  
In order to achieve this, the following columns will need to be appended, using the lookup table and other resources:
- PFR Code
- PFR Name

Futhermore, the columns will have their names changed to be easier and standardised. The data definitions and types will be held in the data dictionary.


***

In [62]:
raw_pop_2022.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   LAD 2023 Code   35672 non-null  object
 1   LAD 2023 Name   35672 non-null  object
 2   LSOA 2021 Code  35672 non-null  object
 3   LSOA 2021 Name  35672 non-null  object
 4   Total           35672 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.4+ MB


In [63]:
raw_pop_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   LAD 2023 Code   35672 non-null  object
 1   LAD 2023 Name   35672 non-null  object
 2   LSOA 2021 Code  35672 non-null  object
 3   LSOA 2021 Name  35672 non-null  object
 4   Total           35672 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.4+ MB


In [64]:
raw_pop_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   LAD 2023 Code   35672 non-null  object
 1   LAD 2023 Name   35672 non-null  object
 2   LSOA 2021 Code  35672 non-null  object
 3   LSOA 2021 Name  35672 non-null  object
 4   Total           35672 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.4+ MB


In [65]:
# Rename current columns
raw_pop_2022 = raw_pop_2022.rename(columns={
    'LAD 2023 Code': 'lad_code',
    'LAD 2023 Name': 'lad_name',
    'LSOA 2021 Code': 'lsoa_code',
    'LSOA 2021 Name': 'lsoa_name',
    'Total': 'population',
})

raw_pop_2023 = raw_pop_2023.rename(columns={
    'LAD 2023 Code': 'lad_code',
    'LAD 2023 Name': 'lad_name',
    'LSOA 2021 Code': 'lsoa_code',
    'LSOA 2021 Name': 'lsoa_name',
    'Total': 'population',
})

raw_pop_2024 = raw_pop_2024.rename(columns={
    'LAD 2023 Code': 'lad_code',
    'LAD 2023 Name': 'lad_name',
    'LSOA 2021 Code': 'lsoa_code',
    'LSOA 2021 Name': 'lsoa_name',
    'Total': 'population',
})

raw_pop_2022.head()

,lad_code,lad_name,lsoa_code,lsoa_name,population
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1876
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1117
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1260
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1635
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,1984


In [66]:
raw_pop_2023.head()

,lad_code,lad_name,lsoa_code,lsoa_name,population
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1925
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1177
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1320
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1670
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2075


In [67]:
raw_pop_2024.head()

,lad_code,lad_name,lsoa_code,lsoa_name,population
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1898
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1247
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1393
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1669
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2303


***
Fix Sheffield and Barnsley LAD Codes, as data is from before 2025
***

In [68]:
# As data comes from before 2025, we need to update the lad_codes for barnsley and sheffield

# Update old Sheffield LAD code to new Sheffield LAD code
raw_pop_2022.loc[raw_pop_2022['lad_code'] == 'E08000016', 'lad_code'] = 'E08000038'

# Update old Barnsley LAD code to new Barnsley LAD code
raw_pop_2022.loc[raw_pop_2022['lad_code'] == 'E08000019', 'lad_code'] = 'E08000039'

raw_pop_2022[raw_pop_2022['lad_code'].isin(['E08000038', 'E08000039'])]

,lad_code,lad_name,lsoa_code,lsoa_name,population
24051,E08000038,Barnsley,E01007317,Barnsley 018A,1532
24052,E08000038,Barnsley,E01007318,Barnsley 018B,1577
24053,E08000038,Barnsley,E01007319,Barnsley 015A,1423
24054,E08000038,Barnsley,E01007320,Barnsley 018C,1741
24055,E08000038,Barnsley,E01007321,Barnsley 015B,1434
...,...,...,...,...,...
24906,E08000039,Sheffield,E01034842,Sheffield 022I,3476
24907,E08000039,Sheffield,E01034843,Sheffield 036F,3013
24908,E08000039,Sheffield,E01034844,Sheffield 074F,2005
24909,E08000039,Sheffield,E01034845,Sheffield 076G,1389


In [69]:
# As data comes from before 2025, we need to update the lad_codes for barnsley and sheffield

# Update old Sheffield LAD code to new Sheffield LAD code
raw_pop_2023.loc[raw_pop_2023['lad_code'] == 'E08000016', 'lad_code'] = 'E08000038'

# Update old Barnsley LAD code to new Barnsley LAD code
raw_pop_2023.loc[raw_pop_2023['lad_code'] == 'E08000019', 'lad_code'] = 'E08000039'

raw_pop_2023[raw_pop_2023['lad_code'].isin(['E08000038', 'E08000039'])]

,lad_code,lad_name,lsoa_code,lsoa_name,population
24051,E08000038,Barnsley,E01007317,Barnsley 018A,1549
24052,E08000038,Barnsley,E01007318,Barnsley 018B,1570
24053,E08000038,Barnsley,E01007319,Barnsley 015A,1441
24054,E08000038,Barnsley,E01007320,Barnsley 018C,1810
24055,E08000038,Barnsley,E01007321,Barnsley 015B,1471
...,...,...,...,...,...
24906,E08000039,Sheffield,E01034842,Sheffield 022I,3573
24907,E08000039,Sheffield,E01034843,Sheffield 036F,3166
24908,E08000039,Sheffield,E01034844,Sheffield 074F,2190
24909,E08000039,Sheffield,E01034845,Sheffield 076G,1429


In [70]:
# As data comes from before 2025, we need to update the lad_codes for barnsley and sheffield

# Update old Sheffield LAD code to new Sheffield LAD code
raw_pop_2024.loc[raw_pop_2024['lad_code'] == 'E08000016', 'lad_code'] = 'E08000038'

# Update old Barnsley LAD code to new Barnsley LAD code
raw_pop_2024.loc[raw_pop_2024['lad_code'] == 'E08000019', 'lad_code'] = 'E08000039'

raw_pop_2024[raw_pop_2024['lad_code'].isin(['E08000038', 'E08000039'])]

,lad_code,lad_name,lsoa_code,lsoa_name,population
24051,E08000038,Barnsley,E01007317,Barnsley 018A,1540
24052,E08000038,Barnsley,E01007318,Barnsley 018B,1584
24053,E08000038,Barnsley,E01007319,Barnsley 015A,1453
24054,E08000038,Barnsley,E01007320,Barnsley 018C,1785
24055,E08000038,Barnsley,E01007321,Barnsley 015B,1438
...,...,...,...,...,...
24906,E08000039,Sheffield,E01034842,Sheffield 022I,3597
24907,E08000039,Sheffield,E01034843,Sheffield 036F,3118
24908,E08000039,Sheffield,E01034844,Sheffield 074F,2185
24909,E08000039,Sheffield,E01034845,Sheffield 076G,1457


***
The data is now fully prepared to be concatinated. We will now create the 2025 and 2026 predicted populations
***

In [71]:
location_cols = ['lsoa_code', 'lsoa_name', 'lad_code', 'lad_name']

pop_growth = raw_pop_2022[location_cols + ['population']].merge(
    raw_pop_2023[location_cols + ['population']],
    on=location_cols,
    how='inner',
    suffixes=('_2022', '_2023')
).merge(
    raw_pop_2024[location_cols + ['population']],
    on=location_cols,
    how='inner'
)

pop_growth = pop_growth.rename(columns={
    'population': 'population_2024'
})

pop_growth.head()

,lsoa_code,lsoa_name,lad_code,lad_name,population_2022,population_2023,population_2024
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,1876,1925,1898
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,1117,1177,1247
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,1260,1320,1393
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,1635,1670,1669
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,1984,2075,2303


In [72]:
# Calculate average annual change
pop_growth['change_2022_2023'] = (
    pop_growth['population_2023'] - pop_growth['population_2022']
)

pop_growth['change_2023_2024'] = (
    pop_growth['population_2024'] - pop_growth['population_2023']
)

pop_growth['avg_annual_change'] = (
    pop_growth[['change_2022_2023', 'change_2023_2024']].mean(axis=1)
)

pop_growth.head()

,lsoa_code,lsoa_name,lad_code,lad_name,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,1876,1925,1898,49,-27,11.0
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,1117,1177,1247,60,70,65.0
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,1260,1320,1393,60,73,66.5
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,1635,1670,1669,35,-1,17.0
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,1984,2075,2303,91,228,159.5


In [73]:
# Add new column for 2025 estimated population
# ASSUMPTION: Growth rate will stay, on average, the same for the next 2 years after the data

pop_growth['population_2025'] = pop_growth['population_2024'] + pop_growth['avg_annual_change']

pop_growth.head()

,lsoa_code,lsoa_name,lad_code,lad_name,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change,population_2025
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,1876,1925,1898,49,-27,11.0,1909.0
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,1117,1177,1247,60,70,65.0,1312.0
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,1260,1320,1393,60,73,66.5,1459.5
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,1635,1670,1669,35,-1,17.0,1686.0
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,1984,2075,2303,91,228,159.5,2462.5


In [74]:
# Add a new columns for 2026 estimated population

pop_growth['population_2026'] = pop_growth['population_2025'] + pop_growth['avg_annual_change']

pop_growth.head()

,lsoa_code,lsoa_name,lad_code,lad_name,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change,population_2025,population_2026
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,1876,1925,1898,49,-27,11.0,1909.0,1920.0
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,1117,1177,1247,60,70,65.0,1312.0,1377.0
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,1260,1320,1393,60,73,66.5,1459.5,1526.0
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,1635,1670,1669,35,-1,17.0,1686.0,1703.0
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,1984,2075,2303,91,228,159.5,2462.5,2622.0


***
There is now columns for the expected population of each LSOA for 2022-26.  
We will now merge with the lookup table in order to get the PFA code and name for each LSOA as well.  
  
Finally, we will then cut down the table to essential columns, and export it to be used in aggregation.  
***

In [75]:
# Merge with lookup table to add PFA stats

pop_growth_pfa = pd.merge(pop_growth,lookup_table,how='left',on=['lsoa_code', 'lsoa_name', 'lad_code', 'lad_name'])

pop_growth_pfa.head()

,lsoa_code,lsoa_name,lad_code,lad_name,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change,population_2025,population_2026,pfa_code,pfa_name
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,1876,1925,1898,49,-27,11.0,1909.0,1920.0,E23000013,Cleveland
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,1117,1177,1247,60,70,65.0,1312.0,1377.0,E23000013,Cleveland
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,1260,1320,1393,60,73,66.5,1459.5,1526.0,E23000013,Cleveland
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,1635,1670,1669,35,-1,17.0,1686.0,1703.0,E23000013,Cleveland
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,1984,2075,2303,91,228,159.5,2462.5,2622.0,E23000013,Cleveland


In [76]:
## Create final database for population data

population_final = pop_growth_pfa[[
    'lsoa_code', 
    'lsoa_name', 
    'lad_code', 
    'lad_name', 
    'pfa_code', 
    'pfa_name', 
    'population_2022', 
    'population_2023', 
    'population_2024', 
    'population_2025', 
    'population_2026'
]]

population_final.head()

,lsoa_code,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,population_2022,population_2023,population_2024,population_2025,population_2026
0,E01011949,Hartlepool 009A,E06000001,Hartlepool,E23000013,Cleveland,1876,1925,1898,1909.0,1920.0
1,E01011950,Hartlepool 008A,E06000001,Hartlepool,E23000013,Cleveland,1117,1177,1247,1312.0,1377.0
2,E01011951,Hartlepool 007A,E06000001,Hartlepool,E23000013,Cleveland,1260,1320,1393,1459.5,1526.0
3,E01011952,Hartlepool 002A,E06000001,Hartlepool,E23000013,Cleveland,1635,1670,1669,1686.0,1703.0
4,E01011953,Hartlepool 002B,E06000001,Hartlepool,E23000013,Cleveland,1984,2075,2303,2462.5,2622.0


***
Final cleanliness check
***

In [77]:
population_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   lsoa_code        35672 non-null  object 
 1   lsoa_name        35672 non-null  object 
 2   lad_code         35672 non-null  object 
 3   lad_name         35672 non-null  object 
 4   pfa_code         35119 non-null  object 
 5   pfa_name         35119 non-null  object 
 6   population_2022  35672 non-null  int64  
 7   population_2023  35672 non-null  int64  
 8   population_2024  35672 non-null  int64  
 9   population_2025  35672 non-null  float64
 10  population_2026  35672 non-null  float64
dtypes: float64(2), int64(3), object(6)
memory usage: 3.0+ MB


***
population_2025 and population_2026 should be integers, they are denoting populations, and you can't have half a person.
***

In [78]:
population_final.loc[:, 'population_2025'] = (population_final['population_2025'].round().astype(int))

population_final.loc[:, 'population_2026'] = (population_final['population_2026'].round().astype(int))

In [79]:
population_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   lsoa_code        35672 non-null  object 
 1   lsoa_name        35672 non-null  object 
 2   lad_code         35672 non-null  object 
 3   lad_name         35672 non-null  object 
 4   pfa_code         35119 non-null  object 
 5   pfa_name         35119 non-null  object 
 6   population_2022  35672 non-null  int64  
 7   population_2023  35672 non-null  int64  
 8   population_2024  35672 non-null  int64  
 9   population_2025  35672 non-null  float64
 10  population_2026  35672 non-null  float64
dtypes: float64(2), int64(3), object(6)
memory usage: 3.0+ MB


In [80]:
# Check for nulls
population_final.isnull().sum()

lsoa_code            0
lsoa_name            0
lad_code             0
lad_name             0
pfa_code           553
pfa_name           553
population_2022      0
population_2023      0
population_2024      0
population_2025      0
population_2026      0
dtype: int64

In [81]:
# Check for nulls
population_final[population_final.isnull().any(axis=1)]['lad_name'].value_counts()

lad_name
Bristol               268
Kingston upon Hull    168
Herefordshire         117
Name: count, dtype: int64

There are missing PFA's for some areas in Bristol, Kingston and Herefordshire.

Luckily, these are not within the data range that this project is analysing, but should be monitored for the case it is expanded.

**Conclusion:** Drop the missing PFA populations.

In [82]:
print(f'number of rows before dropping missing PFAs:{population_final.shape[0]}')

population_final = population_final.dropna()

print(f'number of rows after dropping missing PFAs:{population_final.shape[0]}')

number of rows before dropping missing PFAs:35672
number of rows after dropping missing PFAs:35119


In [83]:
# Check for duplicate data
population_final.duplicated().sum()

np.int64(0)

***
**Population Data is now officially clean!**  
  
Now export data to be used in final aggregation.
***

In [84]:
## Export the code to processed folder as a csv
population_final.to_csv('../Data/Processed/population.csv', index=False)

print(f'lookup-table.csv File successfully created: {Path('../Data/Processed/population.csv').exists()}')

lookup-table.csv File successfully created: True


#### Deprivation

***
This section will create a processed database from the deprivation database. It will have the columns:  
LSOA Code | LSOA Name | LAD Code | LAD Name | PFR code | PFR Name | Deprivation Statistics  
  
In order to achieve this, the following columns must be appended by making use of the lookup table:
- LSOA Name
- LAD Code
- LAD Name
- PFR Code
- PFR Name

Furthermore, the columns will be renamed - and can be defined within the data dictionary later on.
***

In [127]:
depr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33755 entries, 0 to 33754
Data columns (total 6 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   LSOA code (2021)                           33755 non-null  object 
 1   Index of Multiple Deprivation (IMD) Score  33755 non-null  float64
 2   Income Score (rate)                        33755 non-null  float64
 3   Employment Score (rate)                    33755 non-null  float64
 4   Education, Skills and Training Score       33755 non-null  float64
 5   Barriers to Housing and Services Score     33755 non-null  float64
dtypes: float64(5), object(1)
memory usage: 1.5+ MB


In [128]:
# Rename columns
depr = depr.rename(columns={
    'LSOA code (2021)': 'lsoa_code',
    'Index of Multiple Deprivation (IMD) Score': 'imd_score',
    'Income Score (rate)': 'incm_score',
    'Employment Score (rate)': 'empl_score',
    'Education, Skills and Training Score': 'edcn_score',
    'Barriers to Housing and Services Score': 'hous_score'
})

In [131]:
depr.head()

,lsoa_code,imd_score,incm_score,empl_score,edcn_score,hous_score
0,E01000001,8.742,0.013,0.014,0.004,10.950
1,E01000002,4.722,0.018,0.010,0.169,6.703
2,E01000003,9.250,0.107,0.064,3.269,9.735
3,E01000005,19.884,0.211,0.104,17.852,24.623
4,E01000006,25.307,0.343,0.120,25.442,38.025


In [129]:
# No data to deal with 2025 LAD code, as importing directly from lookup table

In [132]:
# Merge with lookup table

depr_final = pd.merge(depr, lookup_table, how='left', on='lsoa_code')

depr_final.head()

,lsoa_code,imd_score,incm_score,empl_score,edcn_score,hous_score,lsoa_name,lad_code,lad_name,pfa_code,pfa_name
0,E01000001,8.742,0.013,0.014,0.004,10.950,City of London 001A,E09000001,City of London,E23000034,"London, City of"
1,E01000002,4.722,0.018,0.010,0.169,6.703,City of London 001B,E09000001,City of London,E23000034,"London, City of"
2,E01000003,9.250,0.107,0.064,3.269,9.735,City of London 001C,E09000001,City of London,E23000034,"London, City of"
3,E01000005,19.884,0.211,0.104,17.852,24.623,City of London 001E,E09000001,City of London,E23000034,"London, City of"
4,E01000006,25.307,0.343,0.120,25.442,38.025,Barking and Dagenham 016A,E09000002,Barking and Dagenham,E23000001,Metropolitan Police


In [134]:
# Reorder columns
cols = ['lsoa_code',
    'lsoa_name',
    'lad_code',
    'lad_name',
    'pfa_code',
    'pfa_name',
    'imd_score',
    'incm_score',
    'empl_score',
    'edcn_score',
    'hous_score']

depr_final = depr_final[cols]

depr_final.head()

,lsoa_code,lsoa_name,lad_code,lad_name,pfa_code,pfa_name,imd_score,incm_score,empl_score,edcn_score,hous_score
0,E01000001,City of London 001A,E09000001,City of London,E23000034,"London, City of",8.742,0.013,0.014,0.004,10.950
1,E01000002,City of London 001B,E09000001,City of London,E23000034,"London, City of",4.722,0.018,0.010,0.169,6.703
2,E01000003,City of London 001C,E09000001,City of London,E23000034,"London, City of",9.250,0.107,0.064,3.269,9.735
3,E01000005,City of London 001E,E09000001,City of London,E23000034,"London, City of",19.884,0.211,0.104,17.852,24.623
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,E23000001,Metropolitan Police,25.307,0.343,0.120,25.442,38.025


In [135]:
# Check database is clean
depr_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33755 entries, 0 to 33754
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   lsoa_code   33755 non-null  object 
 1   lsoa_name   33755 non-null  object 
 2   lad_code    33755 non-null  object 
 3   lad_name    33755 non-null  object 
 4   pfa_code    33755 non-null  object 
 5   pfa_name    33755 non-null  object 
 6   imd_score   33755 non-null  float64
 7   incm_score  33755 non-null  float64
 8   empl_score  33755 non-null  float64
 9   edcn_score  33755 non-null  float64
 10  hous_score  33755 non-null  float64
dtypes: float64(5), object(6)
memory usage: 2.8+ MB


***
All data types are correct
***

In [136]:
# check for null values
depr_final.isnull().sum()

lsoa_code     0
lsoa_name     0
lad_code      0
lad_name      0
pfa_code      0
pfa_name      0
imd_score     0
incm_score    0
empl_score    0
edcn_score    0
hous_score    0
dtype: int64

In [138]:
# check for duplicates
depr_final.duplicated().sum()

np.int64(0)

In [139]:
## Export the code to processed folder as a csv
depr_final.to_csv('../Data/Processed/deprivation-final.csv', index=False)

print(f'deprivation-final.csv File successfully created: {Path('../Data/Processed/deprivation-final.csv').exists()}')

deprivation-final.csv File successfully created: True


#### Crime Severity Weighting

***
This section will create a processed dataset from the crime severity which will find the average weighting for each crime category, to be used in the aggregated dataset in order to aid in visualising where dangerous areas are, rather than where lots of crime happens. It will output in the /Data/Processed folder
***

In [144]:
sev.info()

<class 'pandas.core.frame.DataFrame'>
Index: 247 entries, 0 to 249
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Offence         247 non-null    object
 1   Weight          247 non-null    int64 
 2   Crime Category  247 non-null    object
dtypes: int64(1), object(2)
memory usage: 7.7+ KB


In [146]:
sev = sev.rename(columns={
    'Crime Category': 'crime_cat'
})

## Group Table
sev_by_crime_category = sev.groupby(['crime_cat'])

## Create Columns
num_items = sev_by_crime_category['Weight'].count()

mean_weight = sev_by_crime_category['Weight'].mean()

median_weight = sev_by_crime_category['Weight'].median()

min_weight = sev_by_crime_category['Weight'].min()

max_weight = sev_by_crime_category['Weight'].max()

std_deviation_weight = sev_by_crime_category['Weight'].std()

##Formulate Table
crime_category_severity = pd.DataFrame({
    'num_items': num_items,
    'mean_weight': mean_weight,
    'median_weight': median_weight,
    'min_weight': min_weight,
    'max_weight': max_weight,
    'std_deviation_weight': std_deviation_weight
})

## Visulaise Table
display(crime_category_severity)

,num_items,mean_weight,median_weight,min_weight,max_weight,std_deviation_weight
crime_cat,,,,,,
Bicycle Theft,1,16.000000,16.0,16,16,NaN
Burglary,16,703.250000,438.0,117,2127,697.764765
Criminal damage and arson,12,132.000000,19.0,7,837,255.916890
Drugs,5,105.000000,9.0,3,497,219.157478
No Category,8,280.250000,106.0,106,803,322.648305
Other crime,91,162.813187,86.0,4,4392,459.382603
Other theft,8,143.375000,51.5,7,803,268.795694
Possession of weapons,7,367.142857,75.0,55,1365,490.724101
Public order,8,405.500000,261.0,10,1880,615.286461


***
Looking at these stats, taking the median seems to give a better value, as the skew from large and small data is much less.  
Furthermore, by taking the average - given we are working with large datasets - the skew will become more obvious. This is because lower weighted crimes will be committed more often.  
With median: The more common crimes will be weighted slightly higher than they should, the more dangerous crimes will be rated much lower than they should.  
With mean: The more common crimes will be rated much higher than they should, the more dangerous crimes will be rated lower than they should.  
  
**Assumption:** Median is the best average to use for crime severity weighting.
***

In [ ]:
sev_lookup = (
    sev
    .groupby('crime_cat', as_index=False)
    .agg(avg_weight=('Weight', 'median'))
)

In [ ]:
sev_lookup['crime_cat'] = sev_lookup['crime_cat'].str.lower()

In [160]:
print(sev_lookup.info())

print(sev_lookup.isnull().sum())

print(sev_lookup.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   crime_cat   14 non-null     object 
 1   avg_weight  14 non-null     float64
dtypes: float64(1), object(1)
memory usage: 356.0+ bytes
None
crime_cat     0
avg_weight    0
dtype: int64
0


In [161]:
sev_lookup

,crime_cat,avg_weight
0,bicycle theft,16.0
1,burglary,438.0
2,criminal damage and arson,19.0
3,drugs,9.0
4,no category,106.0
5,other crime,86.0
6,other theft,51.5
7,possession of weapons,75.0
8,public order,261.0
9,robbery,746.0


In [162]:
## Output Final Table to csv
sev_lookup.to_csv('../Data/Processed/crime-category-severity-weighting.csv', index=True)

print(f'File successfully created: {Path('../Data/Processed/crime-category-severity-weighting.csv').exists()}')

File successfully created: True


#### Crime

In [97]:
mssd.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12957 entries, 0 to 13355
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   old_crime_id  12058 non-null  object        
 1   lsoa_code     12957 non-null  object        
 2   lsoa_name     12957 non-null  object        
 3   date          12957 non-null  datetime64[ns]
 4   latitude      12957 non-null  float64       
 5   longitude     12957 non-null  float64       
 6   crime_cat     12957 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(4)
memory usage: 809.8+ KB


***
This section will merge the lookup table to the criem data to get a LAD and PFA for each crime entry.  
It will create a new crime id for each crime entry that is unique and not null.  
The end database will have the layout:  
Crime ID | LSOA Code | LSOA Name | LAD Code | LAD Name | PFA Code | PFA Name | Latitude | Longnitude | Date | Crime Category
***

In [98]:
# Merge with lookup table

mssd_working = pd.merge(mssd, lookup_table, how='left', on=['lsoa_code', 'lsoa_name'])

mssd_working.head()

,old_crime_id,lsoa_code,lsoa_name,date,latitude,longitude,crime_cat,lad_code,lad_name,pfa_code,pfa_name
0,NaN,E01006448,Knowsley 001A,2026-03-01,53.489763,-2.871827,Anti-social behaviour,E08000011,Knowsley,E23000004,Merseyside
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,E01006448,Knowsley 001A,2026-03-01,53.485420,-2.874541,Criminal damage and arson,E08000011,Knowsley,E23000004,Merseyside
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,E01006448,Knowsley 001A,2026-03-01,53.488785,-2.872892,Criminal damage and arson,E08000011,Knowsley,E23000004,Merseyside
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,E01006448,Knowsley 001A,2026-03-01,53.485658,-2.870190,Drugs,E08000011,Knowsley,E23000004,Merseyside
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,E01006448,Knowsley 001A,2026-03-01,53.490168,-2.874261,Other theft,E08000011,Knowsley,E23000004,Merseyside


Now work on creation of new Crime ID. It should have the police force, and date, then an auto-increasing number with a max of 99999, so it can store up to 100,000 crimes per police force area per month.

In [99]:
# Start with pfa name
mssd_working['pfa_short'] = mssd_working['pfa_name'].str.lower().str[:4]

mssd_working.head()

,old_crime_id,lsoa_code,lsoa_name,date,latitude,longitude,crime_cat,lad_code,lad_name,pfa_code,pfa_name,pfa_short
0,NaN,E01006448,Knowsley 001A,2026-03-01,53.489763,-2.871827,Anti-social behaviour,E08000011,Knowsley,E23000004,Merseyside,mers
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,E01006448,Knowsley 001A,2026-03-01,53.485420,-2.874541,Criminal damage and arson,E08000011,Knowsley,E23000004,Merseyside,mers
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,E01006448,Knowsley 001A,2026-03-01,53.488785,-2.872892,Criminal damage and arson,E08000011,Knowsley,E23000004,Merseyside,mers
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,E01006448,Knowsley 001A,2026-03-01,53.485658,-2.870190,Drugs,E08000011,Knowsley,E23000004,Merseyside,mers
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,E01006448,Knowsley 001A,2026-03-01,53.490168,-2.874261,Other theft,E08000011,Knowsley,E23000004,Merseyside,mers


In [100]:
# Next get date
mssd_working['date_str'] = mssd_working['date'].dt.strftime('%Y%m')

mssd_working.head()

,old_crime_id,lsoa_code,lsoa_name,date,latitude,longitude,crime_cat,lad_code,lad_name,pfa_code,pfa_name,pfa_short,date_str
0,NaN,E01006448,Knowsley 001A,2026-03-01,53.489763,-2.871827,Anti-social behaviour,E08000011,Knowsley,E23000004,Merseyside,mers,202603
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,E01006448,Knowsley 001A,2026-03-01,53.485420,-2.874541,Criminal damage and arson,E08000011,Knowsley,E23000004,Merseyside,mers,202603
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,E01006448,Knowsley 001A,2026-03-01,53.488785,-2.872892,Criminal damage and arson,E08000011,Knowsley,E23000004,Merseyside,mers,202603
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,E01006448,Knowsley 001A,2026-03-01,53.485658,-2.870190,Drugs,E08000011,Knowsley,E23000004,Merseyside,mers,202603
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,E01006448,Knowsley 001A,2026-03-01,53.490168,-2.874261,Other theft,E08000011,Knowsley,E23000004,Merseyside,mers,202603


In [101]:
# Now create an increasing number for each entry, which is 5 digits long, and unique.
mssd_working['index_id'] = (mssd_working.index.astype(str).str.zfill(5))

mssd_working.head()

,old_crime_id,lsoa_code,lsoa_name,date,latitude,longitude,crime_cat,lad_code,lad_name,pfa_code,pfa_name,pfa_short,date_str,index_id
0,NaN,E01006448,Knowsley 001A,2026-03-01,53.489763,-2.871827,Anti-social behaviour,E08000011,Knowsley,E23000004,Merseyside,mers,202603,00000
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,E01006448,Knowsley 001A,2026-03-01,53.485420,-2.874541,Criminal damage and arson,E08000011,Knowsley,E23000004,Merseyside,mers,202603,00001
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,E01006448,Knowsley 001A,2026-03-01,53.488785,-2.872892,Criminal damage and arson,E08000011,Knowsley,E23000004,Merseyside,mers,202603,00002
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,E01006448,Knowsley 001A,2026-03-01,53.485658,-2.870190,Drugs,E08000011,Knowsley,E23000004,Merseyside,mers,202603,00003
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,E01006448,Knowsley 001A,2026-03-01,53.490168,-2.874261,Other theft,E08000011,Knowsley,E23000004,Merseyside,mers,202603,00004


In [102]:
# Finally, create a Crime ID using these 3 columns.
mssd_working['crime_id'] = (mssd_working['pfa_short'].str.upper() + mssd_working['date_str'] + '_' + mssd_working['index_id'])

mssd_working.head()

,old_crime_id,lsoa_code,lsoa_name,date,latitude,longitude,crime_cat,lad_code,lad_name,pfa_code,pfa_name,pfa_short,date_str,index_id,crime_id
0,NaN,E01006448,Knowsley 001A,2026-03-01,53.489763,-2.871827,Anti-social behaviour,E08000011,Knowsley,E23000004,Merseyside,mers,202603,00000,MERS202603_00000
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,E01006448,Knowsley 001A,2026-03-01,53.485420,-2.874541,Criminal damage and arson,E08000011,Knowsley,E23000004,Merseyside,mers,202603,00001,MERS202603_00001
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,E01006448,Knowsley 001A,2026-03-01,53.488785,-2.872892,Criminal damage and arson,E08000011,Knowsley,E23000004,Merseyside,mers,202603,00002,MERS202603_00002
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,E01006448,Knowsley 001A,2026-03-01,53.485658,-2.870190,Drugs,E08000011,Knowsley,E23000004,Merseyside,mers,202603,00003,MERS202603_00003
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,E01006448,Knowsley 001A,2026-03-01,53.490168,-2.874261,Other theft,E08000011,Knowsley,E23000004,Merseyside,mers,202603,00004,MERS202603_00004


In [103]:
mssd_working.isnull().sum()

old_crime_id    899
lsoa_code         0
lsoa_name         0
date              0
latitude          0
longitude         0
crime_cat         0
lad_code          0
lad_name          0
pfa_code          0
pfa_name          0
pfa_short         0
date_str          0
index_id          0
crime_id          0
dtype: int64

***
There is now a unique, non-null identifier for each crime. These will stay usnique when aggregating data across years and police forces.
***

Barnsely and Sheffield having incorrect LAD's should be accounted for by using the lookup table.
***

In [104]:
# Create final database schema for the merseyside crime data, which can then be extended to all data

mssd_final = mssd_working[['crime_id', 'date', 'pfa_code', 'pfa_name', 'lsoa_code', 'lsoa_name', 'lad_code', 'lad_name', 'latitude', 'longitude', 'crime_cat']]

mssd_final.head()

,crime_id,date,pfa_code,pfa_name,lsoa_code,lsoa_name,lad_code,lad_name,latitude,longitude,crime_cat
0,MERS202603_00000,2026-03-01,E23000004,Merseyside,E01006448,Knowsley 001A,E08000011,Knowsley,53.489763,-2.871827,Anti-social behaviour
1,MERS202603_00001,2026-03-01,E23000004,Merseyside,E01006448,Knowsley 001A,E08000011,Knowsley,53.485420,-2.874541,Criminal damage and arson
2,MERS202603_00002,2026-03-01,E23000004,Merseyside,E01006448,Knowsley 001A,E08000011,Knowsley,53.488785,-2.872892,Criminal damage and arson
3,MERS202603_00003,2026-03-01,E23000004,Merseyside,E01006448,Knowsley 001A,E08000011,Knowsley,53.485658,-2.870190,Drugs
4,MERS202603_00004,2026-03-01,E23000004,Merseyside,E01006448,Knowsley 001A,E08000011,Knowsley,53.490168,-2.874261,Other theft


***
Final database looks good, final check for cleanliness
***

In [105]:
mssd_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12957 entries, 0 to 12956
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   crime_id   12957 non-null  object        
 1   date       12957 non-null  datetime64[ns]
 2   pfa_code   12957 non-null  object        
 3   pfa_name   12957 non-null  object        
 4   lsoa_code  12957 non-null  object        
 5   lsoa_name  12957 non-null  object        
 6   lad_code   12957 non-null  object        
 7   lad_name   12957 non-null  object        
 8   latitude   12957 non-null  float64       
 9   longitude  12957 non-null  float64       
 10  crime_cat  12957 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(8)
memory usage: 1.1+ MB


In [106]:
mssd_final.isnull().sum()

crime_id     0
date         0
pfa_code     0
pfa_name     0
lsoa_code    0
lsoa_name    0
lad_code     0
lad_name     0
latitude     0
longitude    0
crime_cat    0
dtype: int64

In [107]:
mssd_final.duplicated().sum()

np.int64(0)

fully clean and ready to aggregate.

## Aggregation for Reporting

The end goal of this section is to create the general functions for importing all the data.

It will require a loop that names each individual police force, then each date for the data.  
within the loop, it should import that specific data.  
then clean the data, and check for issues with that.  
then create the same final database for the data, as is shown above.  
Finally, it should append the data to a large database holding all the data, and loop to the next entry.  

In [108]:
# Define function to clean crime data using information from merseyside sample to know what to do.

def CleanCrimeData(raw_crime_data):
    # Drop unimportant columns
    clean_crime_data = raw_crime_data[['Crime ID', 'LSOA code', 'LSOA name', 'Month', 'Latitude', 'Longitude', 'Crime type']]

    # Rename columns
    clean_crime_data = clean_crime_data.rename(columns={
        'Crime ID': 'old_crime_id',
        'Month': 'date',
        'Latitude': 'latitude',
        'Longitude': 'longitude',
        'LSOA code': 'lsoa_code',
        'LSOA name': 'lsoa_name',
        'Crime type': 'crime_cat'
    })

    # Drop duplicate rows
    print(f'row count before dropping: {clean_crime_data.shape[0]}')

    clean_crime_data = clean_crime_data.drop_duplicates()
    print(f'number of duplicates after dropping: {clean_crime_data.duplicated().sum()}')

    print(f'row count after dropping: {clean_crime_data.shape[0]}')


    # Change month to data datatype
    clean_crime_data['date'] = pd.to_datetime(clean_crime_data['date'], format='%Y-%m')
    
    return clean_crime_data


In [109]:
# Define a function to finalise columns of crime data, as defined by the merseyside sample

def FinaliseCrimeData(clean_crime_data):
    # merge with lookup table
    working_crime_data = pd.merge(clean_crime_data, lookup_table, how='left', on=['lsoa_code', 'lsoa_name'])
    
    # make new crime ID
    working_crime_data['pfa_short'] = working_crime_data['pfa_name'].str.lower().str[:4]
    working_crime_data['date_str'] = working_crime_data['date'].dt.strftime('%Y%m')
    working_crime_data['index_id'] = (working_crime_data.index.astype(str).str.zfill(5))
    working_crime_data['crime_id'] = (working_crime_data['pfa_short'].str.upper() + working_crime_data['date_str'] + '_' + working_crime_data['index_id'])

    # create final database
    final_crime_data = working_crime_data[['crime_id', 'date', 'pfa_code', 'pfa_name', 'lsoa_code', 'lsoa_name', 'lad_code', 'lad_name', 'latitude', 'longitude', 'crime_cat']]

    return final_crime_data

In [110]:
# Loop through each police force, then its dates

police_regions = ['merseyside', 'south-yorkshire', 'west-midlands', 'west-yorkshire']
years = ['2023', '2024', '2025', '2026']
months = ['01','02','03','04','05','06','07','08','09','10','11','12']


aggregated_crime_data = pd.DataFrame(columns=['crime_id', 'date', 'pfa_code', 'pfa_name', 'lsoa_code', 'lsoa_name', 'lad_code', 'lad_name', 'latitude', 'longitude', 'crime_cat'])

skipped_data = []
total_rows_dropped = 0

for police_region in police_regions:
    for year in years:
        for month in months:
            file_path = f'../Data/Raw/crime-data/{police_region}/{year}-{month}-{police_region}-street.csv'

            try:
                raw_crime_data = pd.read_csv(file_path)
                print(f'imported {police_region} {year} {month} data')
            except FileNotFoundError:
                print(f'Missing file: {file_path}, skipping...')
                skipped_data.append(f'{police_region}-{year}/{month}, reason: importing')
                continue

            clean_crime_data = CleanCrimeData(raw_crime_data)
            if clean_crime_data is not None:
                print(f'cleaned {police_region} {year} {month} data')
            else:
                print(f'error cleaning {police_region} {year} {month} data, skipping...')
                skipped_data.append(f'{police_region}-{year}/{month}, reason: cleaning')
                total_rows_dropped += raw_crime_data.shape[0]
                continue

            final_crime_data = FinaliseCrimeData(clean_crime_data) # To be defined
            print(f'finalised {police_region} {year} {month} data')

            # Check if clean
            if final_crime_data.duplicated().sum() != 0:
                # Drop duplicate rows
                print(f'row count before dropping: {final_crime_data.shape[0]}')

                final_crime_data = final_crime_data.drop_duplicates()
                print(f'number of duplicates after dropping: {final_crime_data.duplicated().sum()}')

                print(f'row count after dropping: {final_crime_data.shape[0]}')
            if final_crime_data.isnull().sum().sum() != 0:
                # Drop rows with nulls in
                print(f'row count before dropping: {final_crime_data.shape[0]}')

                final_crime_data = final_crime_data.dropna()
                print(f'number of duplicates after dropping: {final_crime_data.isnull().sum().sum()}')

                print(f'row count after dropping: {final_crime_data.shape[0]}')
            
            
            # Append to final data
            aggregated_crime_data = pd.concat([aggregated_crime_data, final_crime_data], ignore_index=True)
            print(f'Added {police_region} {year} {month} data to end data')


Missing file: ../Data/Raw/crime-data/merseyside/2023-01-merseyside-street.csv, skipping...
Missing file: ../Data/Raw/crime-data/merseyside/2023-02-merseyside-street.csv, skipping...
Missing file: ../Data/Raw/crime-data/merseyside/2023-03-merseyside-street.csv, skipping...
imported merseyside 2023 04 data
row count before dropping: 14518
number of duplicates after dropping: 0
row count after dropping: 14076
cleaned merseyside 2023 04 data
finalised merseyside 2023 04 data
row count before dropping: 14076
number of duplicates after dropping: 0
row count after dropping: 13824
row count before dropping: 13824
number of duplicates after dropping: 0
row count after dropping: 13131
Added merseyside 2023 04 data to end data
imported merseyside 2023 05 data
row count before dropping: 15591
number of duplicates after dropping: 0
row count after dropping: 15142
cleaned merseyside 2023 05 data


C:\Users\sam\AppData\Local\Temp\ipykernel_15708\1781477425.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  aggregated_crime_data = pd.concat([aggregated_crime_data, final_crime_data], ignore_index=True)


finalised merseyside 2023 05 data
row count before dropping: 15142
number of duplicates after dropping: 0
row count after dropping: 14949
row count before dropping: 14949
number of duplicates after dropping: 0
row count after dropping: 14346
Added merseyside 2023 05 data to end data
imported merseyside 2023 06 data
row count before dropping: 15025
number of duplicates after dropping: 0
row count after dropping: 14653
cleaned merseyside 2023 06 data
finalised merseyside 2023 06 data
Added merseyside 2023 06 data to end data
imported merseyside 2023 07 data
row count before dropping: 13825
number of duplicates after dropping: 0
row count after dropping: 13460
cleaned merseyside 2023 07 data
finalised merseyside 2023 07 data
Added merseyside 2023 07 data to end data
imported merseyside 2023 08 data
row count before dropping: 13741
number of duplicates after dropping: 0
row count after dropping: 13410
cleaned merseyside 2023 08 data
finalised merseyside 2023 08 data
Added merseyside 2023 0

In [111]:
aggregated_crime_data.sample(10)

,crime_id,date,pfa_code,pfa_name,lsoa_code,lsoa_name,lad_code,lad_name,latitude,longitude,crime_cat
1148875,WEST202311_06332,2023-11-01,E23000014,West Midlands,E01009371,Birmingham 077D,E08000025,Birmingham,52.458973,-1.871682,Violence and sexual offences
299715,MERS202503_11415,2025-03-01,E23000004,Merseyside,E01007194,Wirral 014B,E08000015,Wirral,53.393826,-3.183051,Violence and sexual offences
2274062,WEST202405_15517,2024-05-01,E23000010,West Yorkshire,E01011423,Leeds 047A,E08000035,Leeds,53.817740,-1.493001,Other crime
389331,MERS202510_11496,2025-10-01,E23000004,Merseyside,E01007123,Wirral 011D,E08000015,Wirral,53.401042,-3.061811,Drugs
884688,SOUT202511_12655,2025-11-01,E23000011,South Yorkshire,E01033268,Sheffield 075G,E08000039,Sheffield,53.383549,-1.463989,Public order
1879265,WEST202602_03880,2026-02-01,E23000014,West Midlands,E01034934,Birmingham 052H,E08000025,Birmingham,52.488797,-1.859328,Violence and sexual offences
2270890,WEST202405_12345,2024-05-01,E23000010,West Yorkshire,E01011194,Kirklees 046D,E08000034,Kirklees,53.635260,-1.723100,Other theft
1553101,WEST202501_19552,2025-01-01,E23000014,West Midlands,E01010139,Solihull 006C,E08000029,Solihull,52.487340,-1.737396,Violence and sexual offences
2422362,WEST202411_07096,2024-11-01,E23000010,West Yorkshire,E01011000,Calderdale 010D,E08000033,Calderdale,53.722712,-1.886894,Public order
2741436,WEST202512_08577,2025-12-01,E23000010,West Yorkshire,E01011216,Kirklees 021A,E08000034,Kirklees,53.680972,-1.714564,Violence and sexual offences


In [112]:
aggregated_crime_data.isnull().sum()

crime_id     0
date         0
pfa_code     0
pfa_name     0
lsoa_code    0
lsoa_name    0
lad_code     0
lad_name     0
latitude     0
longitude    0
crime_cat    0
dtype: int64

In [113]:
aggregated_crime_data.duplicated().sum()

np.int64(0)

In [114]:
aggregated_crime_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2820659 entries, 0 to 2820658
Data columns (total 11 columns):
 #   Column     Dtype         
---  ------     -----         
 0   crime_id   object        
 1   date       datetime64[ns]
 2   pfa_code   object        
 3   pfa_name   object        
 4   lsoa_code  object        
 5   lsoa_name  object        
 6   lad_code   object        
 7   lad_name   object        
 8   latitude   float64       
 9   longitude  float64       
 10  crime_cat  object        
dtypes: datetime64[ns](1), float64(2), object(8)
memory usage: 236.7+ MB


In [115]:
aggregated_crime_data.shape

(2820659, 11)

In [116]:
skipped_data

['merseyside-2023/01, reason: importing',
 'merseyside-2023/02, reason: importing',
 'merseyside-2023/03, reason: importing',
 'merseyside-2026/04, reason: importing',
 'merseyside-2026/05, reason: importing',
 'merseyside-2026/06, reason: importing',
 'merseyside-2026/07, reason: importing',
 'merseyside-2026/08, reason: importing',
 'merseyside-2026/09, reason: importing',
 'merseyside-2026/10, reason: importing',
 'merseyside-2026/11, reason: importing',
 'merseyside-2026/12, reason: importing',
 'south-yorkshire-2023/01, reason: importing',
 'south-yorkshire-2023/02, reason: importing',
 'south-yorkshire-2023/03, reason: importing',
 'south-yorkshire-2026/04, reason: importing',
 'south-yorkshire-2026/05, reason: importing',
 'south-yorkshire-2026/06, reason: importing',
 'south-yorkshire-2026/07, reason: importing',
 'south-yorkshire-2026/08, reason: importing',
 'south-yorkshire-2026/09, reason: importing',
 'south-yorkshire-2026/10, reason: importing',
 'south-yorkshire-2026/11,

## All crime data imported
  
Now it's time to use the aggregated data that has been created to add on some extra information to each row.  
Population, depending on the LSOA Code (Land Area)  
Deprivation, depending on LAD Code (Land Area)  
Crime severity, depending on the crime category  
***

In [117]:
# Population

crime_pop = pd.merge(aggregated_crime_data, population_final, how='left', on=['lsoa_code', 'lsoa_name', 'lad_code', 'lad_name', 'pfa_code', 'pfa_name'])

crime_pop['year'] = crime_pop['date'].dt.year

crime_pop['actual_population'] = crime_pop.apply(lambda row: row[f'population_{row["year"]}'], axis=1)

crime_pop.head()

,crime_id,date,pfa_code,pfa_name,lsoa_code,lsoa_name,lad_code,lad_name,latitude,longitude,crime_cat,population_2022,population_2023,population_2024,population_2025,population_2026,year,actual_population
0,CHES202304_00000,2023-04-01,E23000006,Cheshire,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,53.301368,-3.089063,Anti-social behaviour,2048.0,2018.0,2022.0,2009.0,1996.0,2023,2018.0
1,CHES202304_00001,2023-04-01,E23000006,Cheshire,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,53.315654,-3.074956,Vehicle crime,2048.0,2018.0,2022.0,2009.0,1996.0,2023,2018.0
2,CHES202304_00002,2023-04-01,E23000006,Cheshire,E01018570,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,53.300746,-2.960659,Other crime,1534.0,1619.0,1693.0,1772.0,1852.0,2023,1619.0
3,CHES202304_00003,2023-04-01,E23000006,Cheshire,E01012393,Halton 001B,E06000006,Halton,53.389101,-2.746819,Violence and sexual offences,2939.0,2948.0,3031.0,3077.0,3123.0,2023,2948.0
4,CHES202304_00004,2023-04-01,E23000006,Cheshire,E01012376,Halton 002C,E06000006,Halton,53.377272,-2.756590,Anti-social behaviour,2797.0,2822.0,2823.0,2836.0,2849.0,2023,2822.0


In [119]:
crime_pop.shape

(2820659, 18)

In [121]:
depr_final.head()

,lad_code,lad_name,pfa_code,pfa_name,imd_score,incm_score,empl_score,edcn_score,hous_score
0,E09000001,City of London,E23000034,"London, City of",8.742,0.013,0.014,0.004,10.950
1,E09000001,City of London,E23000034,"London, City of",4.722,0.018,0.010,0.169,6.703
2,E09000001,City of London,E23000034,"London, City of",9.250,0.107,0.064,3.269,9.735
3,E09000001,City of London,E23000034,"London, City of",19.884,0.211,0.104,17.852,24.623
4,E09000002,Barking and Dagenham,E23000001,Metropolitan Police,25.307,0.343,0.120,25.442,38.025


In [140]:
# Deprivation

crime_pop_depr = pd.merge(
    crime_pop[['crime_id', 'date', 'pfa_code', 'pfa_name', 'lsoa_code', 'lsoa_name', 'lad_code', 'lad_name', 'latitude', 'longitude', 'crime_cat', 'actual_population']],
    depr_final, 
    how='left', 
    on=['lsoa_code', 'lsoa_name', 'lad_code', 'lad_name', 'pfa_code', 'pfa_name']
)

crime_pop_depr.head()

,crime_id,date,pfa_code,pfa_name,lsoa_code,lsoa_name,lad_code,lad_name,latitude,longitude,crime_cat,actual_population,imd_score,incm_score,empl_score,edcn_score,hous_score
0,CHES202304_00000,2023-04-01,E23000006,Cheshire,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,53.301368,-3.089063,Anti-social behaviour,2018.0,4.137,0.050,0.049,0.889,18.163
1,CHES202304_00001,2023-04-01,E23000006,Cheshire,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,53.315654,-3.074956,Vehicle crime,2018.0,4.137,0.050,0.049,0.889,18.163
2,CHES202304_00002,2023-04-01,E23000006,Cheshire,E01018570,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,53.300746,-2.960659,Other crime,1619.0,10.524,0.087,0.060,4.840,26.361
3,CHES202304_00003,2023-04-01,E23000006,Cheshire,E01012393,Halton 001B,E06000006,Halton,53.389101,-2.746819,Violence and sexual offences,2948.0,4.483,0.051,0.051,2.076,16.072
4,CHES202304_00004,2023-04-01,E23000006,Cheshire,E01012376,Halton 002C,E06000006,Halton,53.377272,-2.756590,Anti-social behaviour,2822.0,5.531,0.053,0.053,3.032,21.859


In [141]:
crime_pop_depr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2820659 entries, 0 to 2820658
Data columns (total 17 columns):
 #   Column             Dtype         
---  ------             -----         
 0   crime_id           object        
 1   date               datetime64[ns]
 2   pfa_code           object        
 3   pfa_name           object        
 4   lsoa_code          object        
 5   lsoa_name          object        
 6   lad_code           object        
 7   lad_name           object        
 8   latitude           float64       
 9   longitude          float64       
 10  crime_cat          object        
 11  actual_population  float64       
 12  imd_score          float64       
 13  incm_score         float64       
 14  empl_score         float64       
 15  edcn_score         float64       
 16  hous_score         float64       
dtypes: datetime64[ns](1), float64(8), object(8)
memory usage: 365.8+ MB


In [168]:
sev_lookup

,crime_cat,avg_weight
0,bicycle theft,16.0
1,burglary,438.0
2,criminal damage and arson,19.0
3,drugs,9.0
4,no category,106.0
5,other crime,86.0
6,other theft,51.5
7,possession of weapons,75.0
8,public order,261.0
9,robbery,746.0


In [164]:
crime_pop_depr['crime_cat'] = crime_pop_depr['crime_cat'].str.lower()

In [165]:
# Crime severity

crime_pop_depr_sev = pd.merge(crime_pop_depr, sev_lookup, how='left', on='crime_cat')

crime_pop_depr_sev.head()

,crime_id,date,pfa_code,pfa_name,lsoa_code,lsoa_name,lad_code,lad_name,latitude,longitude,crime_cat,actual_population,imd_score,incm_score,empl_score,edcn_score,hous_score,avg_weight
0,CHES202304_00000,2023-04-01,E23000006,Cheshire,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,53.301368,-3.089063,anti-social behaviour,2018.0,4.137,0.050,0.049,0.889,18.163,NaN
1,CHES202304_00001,2023-04-01,E23000006,Cheshire,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,53.315654,-3.074956,vehicle crime,2018.0,4.137,0.050,0.049,0.889,18.163,41.0
2,CHES202304_00002,2023-04-01,E23000006,Cheshire,E01018570,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,53.300746,-2.960659,other crime,1619.0,10.524,0.087,0.060,4.840,26.361,86.0
3,CHES202304_00003,2023-04-01,E23000006,Cheshire,E01012393,Halton 001B,E06000006,Halton,53.389101,-2.746819,violence and sexual offences,2948.0,4.483,0.051,0.051,2.076,16.072,709.0
4,CHES202304_00004,2023-04-01,E23000006,Cheshire,E01012376,Halton 002C,E06000006,Halton,53.377272,-2.756590,anti-social behaviour,2822.0,5.531,0.053,0.053,3.032,21.859,NaN


In [166]:
crime_pop_depr_sev[crime_pop_depr_sev['avg_weight'].isna()]['crime_cat'].unique()

array(['anti-social behaviour'], dtype=object)

***
Only Anti-social behaviour is null crime category.  
The crime severity weighting is based off of the severity of court rulings coming from crimes. As ASB often goes unpunished, it does not have a crime severity weighting score.  
**assumption:** Crime severity for ASB is 1. This is a low enough number to barely weigh in to crime severity, but does not leave it unnoticed.
***

In [169]:
crime_pop_depr_sev['avg_weight'] = crime_pop_depr_sev['avg_weight'].fillna(1)

In [170]:
crime_pop_depr_sev.head()

,crime_id,date,pfa_code,pfa_name,lsoa_code,lsoa_name,lad_code,lad_name,latitude,longitude,crime_cat,actual_population,imd_score,incm_score,empl_score,edcn_score,hous_score,avg_weight
0,CHES202304_00000,2023-04-01,E23000006,Cheshire,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,53.301368,-3.089063,anti-social behaviour,2018.0,4.137,0.050,0.049,0.889,18.163,1.0
1,CHES202304_00001,2023-04-01,E23000006,Cheshire,E01018537,Cheshire West and Chester 001D,E06000050,Cheshire West and Chester,53.315654,-3.074956,vehicle crime,2018.0,4.137,0.050,0.049,0.889,18.163,41.0
2,CHES202304_00002,2023-04-01,E23000006,Cheshire,E01018570,Cheshire West and Chester 004C,E06000050,Cheshire West and Chester,53.300746,-2.960659,other crime,1619.0,10.524,0.087,0.060,4.840,26.361,86.0
3,CHES202304_00003,2023-04-01,E23000006,Cheshire,E01012393,Halton 001B,E06000006,Halton,53.389101,-2.746819,violence and sexual offences,2948.0,4.483,0.051,0.051,2.076,16.072,709.0
4,CHES202304_00004,2023-04-01,E23000006,Cheshire,E01012376,Halton 002C,E06000006,Halton,53.377272,-2.756590,anti-social behaviour,2822.0,5.531,0.053,0.053,3.032,21.859,1.0


***
## Final Aggregated Database Completed

**final cleaning will now take place**
***

In [172]:
# Check overall structure and data types
crime_pop_depr_sev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2820659 entries, 0 to 2820658
Data columns (total 18 columns):
 #   Column             Dtype         
---  ------             -----         
 0   crime_id           object        
 1   date               datetime64[ns]
 2   pfa_code           object        
 3   pfa_name           object        
 4   lsoa_code          object        
 5   lsoa_name          object        
 6   lad_code           object        
 7   lad_name           object        
 8   latitude           float64       
 9   longitude          float64       
 10  crime_cat          object        
 11  actual_population  float64       
 12  imd_score          float64       
 13  incm_score         float64       
 14  empl_score         float64       
 15  edcn_score         float64       
 16  hous_score         float64       
 17  avg_weight         float64       
dtypes: datetime64[ns](1), float64(9), object(8)
memory usage: 387.4+ MB


Population should be held as an integer, not as a float.
Every other datatype is correct.

In [171]:
crime_pop_depr_sev.isnull().sum()

crime_id              0
date                  0
pfa_code              0
pfa_name              0
lsoa_code             0
lsoa_name             0
lad_code              0
lad_name              0
latitude              0
longitude             0
crime_cat             0
actual_population    10
imd_score             4
incm_score            4
empl_score            4
edcn_score            4
hous_score            4
avg_weight            0
dtype: int64

Drop any row with null values

In [176]:
crime_pop_depr_sev.duplicated().sum()

np.int64(0)

no duplicates!

In [178]:
# firstly, drop null values
print(f'number of rows before dropping nulls: {crime_pop_depr_sev.shape[0]}')

crime_pop_depr_sev = crime_pop_depr_sev.dropna()

print(f'number of rows after dropping nulls: {crime_pop_depr_sev.shape[0]}')

number of rows before dropping nulls: 2820645
number of rows after dropping nulls: 2820645


In [ ]:
# Change population to integer
crime_pop_depr_sev['actual_population'] = (
    crime_pop_depr_sev['actual_population']
    .round()
    .astype(int)
)

crime_pop_depr_sev.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2820645 entries, 0 to 2820658
Data columns (total 18 columns):
 #   Column             Dtype         
---  ------             -----         
 0   crime_id           object        
 1   date               datetime64[ns]
 2   pfa_code           object        
 3   pfa_name           object        
 4   lsoa_code          object        
 5   lsoa_name          object        
 6   lad_code           object        
 7   lad_name           object        
 8   latitude           float64       
 9   longitude          float64       
 10  crime_cat          object        
 11  actual_population  int64         
 12  imd_score          float64       
 13  incm_score         float64       
 14  empl_score         float64       
 15  edcn_score         float64       
 16  hous_score         float64       
 17  avg_weight         float64       
dtypes: datetime64[ns](1), float64(8), int64(1), object(8)
memory usage: 408.9+ MB


In [181]:
# Rename columns
crime_pop_depr_sev = crime_pop_depr_sev.rename(columns={
    'actual_population': 'population_lsoa',
    'avg_weight': 'crime_sev_weight'
})

***
### Aggregated Crime Database Completed and Cleaned
***

## Export

In [182]:
# Final check
crime_pop_depr_sev.sample(10)

,crime_id,date,pfa_code,pfa_name,lsoa_code,lsoa_name,lad_code,lad_name,latitude,longitude,crime_cat,population_lsoa,imd_score,incm_score,empl_score,edcn_score,hous_score,crime_sev_weight
51611,MERS202307_09481,2023-07-01,E23000004,Merseyside,E01006836,St. Helens 005A,E08000013,St. Helens,53.471454,-2.646826,violence and sexual offences,1475,30.575,0.307,0.218,33.047,11.953,709.0
2490205,WEST202502_04842,2025-02-01,E23000010,West Yorkshire,E01033693,Bradford 064A,E08000032,Bradford,53.794297,-1.748195,shoplifting,3364,60.742,0.539,0.270,64.924,29.499,13.0
1802206,WEST202511_00924,2025-11-01,E23000014,West Midlands,E01009406,Birmingham 020D,E08000025,Birmingham,52.530562,-1.858059,violence and sexual offences,1912,41.544,0.430,0.259,29.953,21.350,709.0
2242488,WEST202404_08650,2024-04-01,E23000010,West Yorkshire,E01011250,Kirklees 009D,E08000034,Kirklees,53.708452,-1.690245,violence and sexual offences,1814,44.188,0.418,0.242,42.336,15.082,709.0
210229,MERS202408_04231,2024-08-01,E23000004,Merseyside,E01034402,Liverpool 037G,E08000012,Liverpool,53.398736,-2.978281,anti-social behaviour,1566,25.181,0.163,0.067,26.458,25.388,1.0
2579947,WEST202505_22913,2025-05-01,E23000010,West Yorkshire,E01011836,Wakefield 012B,E08000036,Wakefield,53.696895,-1.300150,other theft,1430,20.287,0.226,0.131,26.292,11.963,51.5
1382909,WEST202407_16552,2024-07-01,E23000014,West Midlands,E01009708,Coventry 033F,E08000026,Coventry,52.405675,-1.453653,criminal damage and arson,1436,13.881,0.144,0.100,21.659,13.488,19.0
611305,SOUT202403_09329,2024-03-01,E23000011,South Yorkshire,E01007913,Sheffield 018C,E08000039,Sheffield,53.414961,-1.411893,shoplifting,1750,65.514,0.678,0.299,81.612,28.544,13.0
1275907,WEST202403_25149,2024-03-01,E23000014,West Midlands,E01010335,Walsall 037A,E08000030,Walsall,52.561876,-1.978700,violence and sexual offences,1726,47.025,0.574,0.279,46.306,16.296,709.0
1295509,WEST202404_16842,2024-04-01,E23000014,West Midlands,E01009826,Dudley 019B,E08000027,Dudley,52.485232,-2.171100,violence and sexual offences,1447,4.649,0.092,0.057,5.535,14.577,709.0


In [184]:
# Export to csv file

## Output Final Table to csv
crime_pop_depr_sev.to_csv('../Data/Output/crime-data-final.csv', index=True)

print(f'File successfully created: {Path('../Data/Output/crime-data-final.csv').exists()}')

File successfully created: True
